# Rayleigh numbers

## Load the simulations

In [ ]:
from functools import partial
import numpy as np
from lucifex.fdm import GridFunctionSeries, NPyConstantSeries
from lucifex.fem import refine_grid_function, grid_average
from lucifex.viz import (
    plot_colormap, plot_line, save_figure,
    plot_colormap_multifigure, plot_line_multifigure,
)
from lucifex.utils.array_utils import as_index
from lucifex.io import create_dir_path, find_dir_paths
from lucifex.utils.py_utils import FrozenDict
from lucifex.sim import GridSimulationFromNPZ
from crocodil.dns.system_a import (
    SYSTEM_A_REFERENCE, mass_dissolved_asymptote, mass_capillary_asymptote,
    mass_dissolved_initial, mass_capillary_initial,
)

PARAM_NAME = 'Ra'PARAM_TEX = 'Ra'save_figure = partial(save_figure, prefix=PARAM_NAME)

NUMERICAL_PARAMS = FrozenDict(
    c_stabilization=None,
    c_limits=True,
)
DIR_ROOT = create_dir_path(
    NUMERICAL_PARAMS, 
    dir_root='./',
    dir_prefix='data', 
    dir_params=NUMERICAL_PARAMS.keys(), 
)
DIR_FIGS = f'{DIR_ROOT}/figures'
N_STOP = 50_000

sim_dir_paths = find_dir_paths(
    DIR_ROOT, 
    include=f'n_stop={N_STOP}_*',
    contains=('CHECKPOINT.h5', 'c.npz'),
)
print('Before parameter filtering')
for i in sim_dir_paths: print(i)

simulations = GridSimulationFromNPZ.dict_from_dir_paths(
    PARAM_NAME, 
    sim_dir_paths,
    ('c', 's'),
    ('f', 'mD', 'mC'),
    SYSTEM_A_REFERENCE,
    lazy=False,
)

print('After parameter filtering')
for i in simulations.values(): print(i.dir_path)

epsilon, zeta0, sr, cr, aspect = SYSTEM_A_REFERENCE['epsilon', 'zeta0', 'sr', 'cr', 'aspect']
Ly = 1.0
Lx = aspect * Ly

In [ ]:
function_series = 'c'
for param_value, sim in simulations.items():
    time_series = sim[function_series].time_series
    print(f'{PARAM_NAME} = {param_value}')
    print(f"n_time = {len(time_series)} t_min = {np.min(time_series)}  t_max = {np.max(time_series)}")

## Flux & mass

In [ ]:
f_lines, mC_lines, mD_lines = [], [], []
legend_labels = []
for param_value, sim in simulations.items():
    f, mC, mD = sim['f', 'mC', 'mD']
    fZeta0, fZetaPlus, fZetaMinus = f.split()
    f_lines.append((fZeta0.time_series, [np.sum(i) for i in fZeta0.value_series]))
    mC_lines.append((mC.time_series, mC.value_series))
    mD_lines.append((mD.time_series, mD.value_series))
    legend_labels.append(param_value)

line_kws = dict(
    cyc='black',
    x_label='$t$',
    legend_labels=legend_labels,
    legend_title=f'${PARAM_TEX}$',
)
hline_kws = dict(
    xmin=0, xmax=120.0, 
    linestyles=(5, (10, 3)), colors='gray', linewidths=0.75,
)

fig, ax = plot_line(f_lines, y_label='$F(y=\zeta_0)$', x_lims=(0, 10.0), **line_kws)
if DIR_FIGS: save_figure('f(t)_early', DIR_FIGS)(fig, pickle=True)

fig, ax = plot_line(f_lines, y_label='$F(y=\zeta_0)$', x_lims=(0, 60.0), **line_kws)
save_figure('f(t)_late', DIR_FIGS)(fig, pickle=True)

mC_initial = mass_capillary_initial(epsilon, zeta0, sr, Lx, Ly)
mD_initial = mass_dissolved_initial(zeta0, sr, cr, Lx, Ly)
m_initial = mC_initial + mD_initial

mC_asymp = mass_capillary_asymptote(m_initial, epsilon, Lx, Ly)
mD_asymp = mass_dissolved_asymptote(m_initial, epsilon, Lx, Ly)

fig, ax = plot_line(mC_lines, y_label='$m_C$', **line_kws)
ax.hlines([mC_initial, mC_asymp], **hline_kws)
if DIR_FIGS: save_figure('mC(t)', DIR_FIGS)(fig, pickle=True)

fig, ax = plot_line(mD_lines, y_label='$m_D$', **line_kws)
ax.hlines([mD_initial, mD_asymp], **hline_kws)
if DIR_FIGS: save_figure('mD(t)', DIR_FIGS)(fig, pickle=True)

## Horizontal averages

In [ ]:
slc = slice(0, None, 10)
titles = []
sAvgX_lines, sAvgX_legend_labels = [], []
cAvgX_lines, cAvgX_legend_labels = [], []

for param_value, sim in simulations.items():
    titles.append(f'\n${PARAM_TEX}={param_value}$')
    s, c = sim['s', 'c']
    sAvgX = GridFunctionSeries(
        [
            grid_average(refine_grid_function(i, (1, 10)), 'x')
            for i in s.series[slc]
        ], 
        s.time_series[slc], 
        'sAvgX',
    )
    sAvgX_legend_labels.append((min(sAvgX.time_series), max(sAvgX.time_series)))
    sAvgX_lines.append(sAvgX.series)
    cAvgX = GridFunctionSeries(
        [grid_average(use_cache=True)(i, 'x') for i in c.series[slc]], c.time_series[slc], 'cAvgX',
    )
    cAvgX_legend_labels.append((min(cAvgX.time_series), max(cAvgX.time_series)))
    cAvgX_lines.append(cAvgX.series)


multiline_kws = dict(
    cyc='jet',
    title=titles, 
    legend_title='$t$',
    flip=True,
    y_label='$y$',
)

fig, *_  = plot_line_multifigure(n_cols=1, cbars=True)(
    cAvgX_lines, 
    legend_labels=cAvgX_legend_labels,
    x_label='$\langle c\\rangle_x$', 
    y_lims=(0, Ly),
    **multiline_kws,
)

fig, *_  = plot_line_multifigure(n_cols=1, cbars=True)(
    sAvgX_lines, 
    legend_labels=sAvgX_legend_labels,
    x_label='$\langle s\\rangle_x$', 
    y_lims=(zeta0 * Ly, Ly),
    **multiline_kws,
)

## Subdomain averages

In [ ]:
legend_labels = []
cPlus_lines, sPlus_lines, cMinus_lines = [], [], []

for param_value, sim in simulations.items():
    legend_labels.append(param_value)
    s, c = sim['s', 'c']
    y_axis = c.mesh.y_axis
    zeta0_index = as_index(y_axis, zeta0)
    slcPlus = slice(zeta0_index, None)
    slcMinus = slice(0, zeta0_index)
    cPlus = NPyConstantSeries(
        grid_average(c.series, ('x', 'y'), (':', slcPlus)), c.time_series, 'cPlus',
    )
    sPlus = NPyConstantSeries(
        grid_average(s.series, ('x', 'y'), (':', slcPlus)), s.time_series, 'sPlus',
    )
    cMinus = NPyConstantSeries(
        grid_average(c.series, ('x', 'y'), (':', slcMinus)), c.time_series, 'cMinus',
    )
    cPlus_lines.append((cPlus.time_series, cPlus.value_series))
    sPlus_lines.append((sPlus.time_series, sPlus.value_series))
    cMinus_lines.append((cMinus.time_series, cMinus.value_series))

line_kws = dict(
    cyc='black',
    x_label='$t$',
    legend_labels=legend_labels,
    legend_title=f'${PARAM_TEX}$',
)

fig, ax = plot_line(
    cPlus_lines,
    y_label='$c^+$',
    **line_kws,
)
if DIR_FIGS: save_figure('cPlus(t)', DIR_FIGS)(fig, pickle=True)

fig, ax = plot_line(
    sPlus_lines,
    y_label='$s^+$',
    **line_kws,
)
if DIR_FIGS: save_figure('sPlus(t)', DIR_FIGS)(fig, pickle=True)

fig, ax = plot_line(
    cMinus_lines,
    y_label='$c^-$',
    **line_kws,
)
if DIR_FIGS: save_figure('cMinus(t)', DIR_FIGS)(fig, pickle=True)

## Visualization

In [ ]:
c_t_target = 10.0
s_t_target = 10.0

c_cmaps, c_titles = [], []
s_cmaps, s_titles = [], []

for param_value, sim in simulations.items():
    c = sim['c']
    c_time_index = as_index(c.time_series, c_t_target)
    c_cmaps.append(c.series[c_time_index])
    c_titles.append(f'${PARAM_TEX}={param_value}$\n $c(t={c.time_series[c_time_index]:.3f})$')
    s = sim['s']
    s_time_index = as_index(s.time_series, s_t_target)
    s_cmaps.append(refine_grid_function(use_cache=True)(s.series[s_time_index], (1, 10)))
    s_titles.append(f'${PARAM_TEX}={param_value}$\n $s(t={s.time_series[s_time_index]:.3f})$')


fig, *_  = plot_colormap_multifigure(n_cols=1, cbars=(0,1))(
    c_cmaps, titles=c_titles,
)
if DIR_FIGS: save_figure(f'c(x,y,t={c_t_target})', DIR_FIGS)(fig, pickle=True)

fig, *_  = plot_colormap_multifigure(n_cols=1, cbars=(0, sr))(
    s_cmaps, titles=s_titles, y_lims=(zeta0 * Ly, Ly), aspect='auto',
)
if DIR_FIGS: save_figure(f's(x,y,t={s_t_target})', DIR_FIGS)(fig, pickle=True)